# Практикум 15. Диалоговые системы: интенты, слоты, сценарии и генеративные модели

В лаборатории 15 вы видели, как реплика пользователя превращается в интент и слоты, как фрейм заполняется по ходу диалога и чем сценарная система отличается от генеративной. Здесь соберём такую систему в коде (домен: покупка билета на поезд и бронирование столика):

1. классификатор интентов: softmax-регрессия с нуля на numpy, затем scikit-learn;
2. заполнение слотов в разметке BIO: правила против CRF (`sklearn-crfsuite`);
3. фреймовый менеджер диалога (конечный автомат) и шаблонная генерация ответа;
4. игрушечная генеративная модель ответов: почему она «выдумывает» значения слотов и как помогает делексикализация.

*Практикум к лаборатории 15 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Данные: реплики с разметкой

Каждая реплика размечена интентом (что хочет пользователь) и слотами (параметры запроса). Слоты записаны прямо в тексте в формате `[значение](слот)` — так размечают данные во многих фреймворках для чат-ботов. Функция `parse_markup` превращает разметку в токены и BIO-теги: `B-` — первое слово слота, `I-` — продолжение, `O` — вне слота.

Всего 60 обучающих реплик (6 интентов по 10) и 12 тестовых. Если хотите попробовать свои данные, загрузите файл через панель «Файлы» и прочитайте его в `train_data`.

In [ ]:
!pip install -q sklearn-crfsuite

In [ ]:
# Разметка слотов прямо в тексте: [значение](слот). Слоты: from, to, date, time, count
train_data = {
    "buy_ticket": [
        "Хочу купить билет из [Ташкента](from) в [Самарканд](to)",
        "Мне нужен билет до [Бухары](to) на [завтра](date)",
        "Купите мне [два](count) билета в [Самарканд](to) на [пятницу](date)",
        "Билет из [Самарканда](from) в [Ташкент](to) на [10 мая](date), пожалуйста",
        "Забронируйте [три](count) места на поезд до [Хивы](to)",
        "Хочу уехать в [Бухару](to) [послезавтра](date) [утром](time)",
        "Есть билеты из [Бухары](from) в [Ташкент](to) на [субботу](date)?",
        "Оформите билет на Афросиаб до [Самарканда](to)",
        "Нужно [два](count) билета на [завтра](date) на [8:00](time)",
        "Купить билет [Ташкент](from) – [Нукус](to)",
    ],
    "schedule": [
        "Когда отправляется поезд в [Самарканд](to)?",
        "Во сколько поезд из [Ташкента](from) в [Бухару](to)?",
        "Какое расписание поездов до [Хивы](to) на [завтра](date)?",
        "Подскажите время отправления Афросиаба",
        "Есть ли поезд в [Карши](to) [вечером](time)?",
        "Сколько едет поезд до [Бухары](to)?",
        "Когда ближайший поезд из [Самарканда](from)?",
        "Покажите расписание на [пятницу](date)",
        "Во сколько прибывает поезд в [Ташкент](to)?",
        "Какие поезда идут в [Нукус](to) [утром](time)?",
    ],
    "book_table": [
        "Хочу забронировать столик на [четверых](count)",
        "Забронируйте стол на [сегодня](date) на [19:00](time)",
        "Нужен столик на [двоих](count) [завтра](date) [вечером](time)",
        "Можно заказать стол в ресторане на [субботу](date)?",
        "Зарезервируйте столик на [пять](count) человек",
        "Хочу поужинать у вас [сегодня](date) в [20:00](time)",
        "Будет свободный столик на [троих](count) в [18:30](time)?",
        "Запишите нас на ужин в [пятницу](date)",
        "Забронируйте, пожалуйста, стол у окна",
        "Столик на [шесть](count) человек на [завтра](date)",
    ],
    "cancel": [
        "Отмените мой заказ",
        "Хочу отменить бронь",
        "Отмена, я передумал",
        "Не нужно бронировать столик",
        "Верните деньги за билет",
        "Я хочу сдать билет",
        "Отменить бронирование столика",
        "Отменяю поездку",
        "Аннулируйте заказ, пожалуйста",
        "Планы изменились, отмените всё",
    ],
    "greeting": [
        "Здравствуйте",
        "Привет",
        "Добрый день",
        "Доброе утро",
        "Добрый вечер",
        "Здравствуйте, мне нужна помощь",
        "Привет, бот",
        "Алло, вы меня слышите?",
        "Приветствую",
        "Здравствуйте, можно вопрос?",
    ],
    "thanks_bye": [
        "Спасибо",
        "Спасибо, до свидания",
        "Большое спасибо за помощь",
        "Всё, пока",
        "До свидания",
        "Благодарю",
        "Спасибо, это всё",
        "Пока",
        "Всего доброго",
        "Отлично, спасибо",
    ],
}

test_data = [
    ("Мне нужен билет из [Ташкента](from) в [Хиву](to)", "buy_ticket"),
    ("Купите [один](count) билет до [Самарканда](to) на [понедельник](date)", "buy_ticket"),
    ("Хочу поехать в [Андижан](to) [завтра](date) [вечером](time)", "buy_ticket"),
    ("Во сколько уходит поезд до [Нукуса](to)?", "schedule"),
    ("Расписание поездов из [Бухары](from) на [завтра](date)", "schedule"),
    ("Столик на [двоих](count) на [субботу](date), пожалуйста", "book_table"),
    ("Можно забронировать стол на [19:30](time)?", "book_table"),
    ("Привет, нужен столик [сегодня](date) [вечером](time)", "book_table"),
    ("Я передумал, отмените бронь", "cancel"),
    ("Сдать билет на поезд", "cancel"),
    ("Добрый день, помогите мне", "greeting"),
    ("Спасибо большое, до свидания", "thanks_bye"),
]


import re
import numpy as np

TOKEN_RE = r"\d+(?::\d+)?(?:-[^\W\d_]+)?|[^\W\d_]+(?:-[^\W\d_]+)*"
MARK_RE = r"\[([^\]]+)\]\((\w+)\)"

def tokenize(text):
    return re.findall(TOKEN_RE, text)

def parse_markup(marked):
    # "[Ташкента](from) ..." -> (токены, BIO-теги)
    tokens, tags, pos = [], [], 0
    for m in re.finditer(MARK_RE, marked):
        for t in tokenize(marked[pos:m.start()]):
            tokens.append(t); tags.append("O")
        for k, t in enumerate(tokenize(m.group(1))):
            tokens.append(t); tags.append(("B-" if k == 0 else "I-") + m.group(2))
        pos = m.end()
    for t in tokenize(marked[pos:]):
        tokens.append(t); tags.append("O")
    return tokens, tags

def plain(marked):
    return re.sub(MARK_RE, r"\1", marked)

train = [(plain(s), intent) for intent, sents in train_data.items() for s in sents]
test = [(plain(s), intent) for s, intent in test_data]
train_bio = [parse_markup(s) for sents in train_data.values() for s in sents]
test_bio = [parse_markup(s) for s, _ in test_data]
INTENTS = sorted(train_data)

print(len(train), "обучающих реплик,", len(test), "тестовых,", "интенты:", INTENTS)
example = test_data[0][0]
print("\n" + example)
for tok, tag in zip(*parse_markup(example)):
    print(f"   {tok:<14} {tag}")

## 2. Классификация интентов

### С нуля: softmax-регрессия

Реплика → вектор «мешка слов» $x$ (1, если слово есть в реплике). Модель: $P(c\mid x)=\text{softmax}(xW+b)_c$. Обучаем градиентным спуском по кросс-энтропии; градиент по логитам простой: $P-Y$, где $Y$ — one-hot правильного интента. Слова, которых не было в обучении, модель просто не видит.

In [ ]:
def bag(text):
    return sorted(set(w.lower() for w in tokenize(text)))

vocab = sorted({w for s, _ in train for w in bag(s)})
w2i = {w: i for i, w in enumerate(vocab)}

def featurize(texts):
    X = np.zeros((len(texts), len(vocab)))
    for r, s in enumerate(texts):
        for w in bag(s):
            if w in w2i:                 # незнакомые слова пропускаем
                X[r, w2i[w]] = 1.0
    return X

def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

X_tr = featurize([s for s, _ in train])
y_tr = np.array([INTENTS.index(c) for _, c in train])
X_te = featurize([s for s, _ in test])
y_te = np.array([INTENTS.index(c) for _, c in test])
Y = np.eye(len(INTENTS))[y_tr]                      # one-hot

rng = np.random.default_rng(0)
W = rng.normal(0, 0.01, size=(len(vocab), len(INTENTS)))
b = np.zeros(len(INTENTS))
lr, lam = 0.5, 1e-3
for epoch in range(301):
    P = softmax(X_tr @ W + b)
    loss = -np.mean(np.log(P[np.arange(len(y_tr)), y_tr])) + lam / 2 * (W ** 2).sum()
    G = (P - Y) / len(y_tr)                          # градиент кросс-энтропии по логитам
    W -= lr * (X_tr.T @ G + lam * W)
    b -= lr * G.sum(axis=0)
    if epoch % 100 == 0:
        print(f"эпоха {epoch:3d}  loss = {loss:.3f}")

P_te = softmax(X_te @ W + b)
print(f"\nточность (с нуля, слова): {np.mean(P_te.argmax(1) == y_te):.2f}\n")
for (s, gold), p in zip(test, P_te):
    k = p.argmax()
    mark = "" if INTENTS[k] == gold else "   <-- " + gold
    print(f"{s:<52} {INTENTS[k]:<11} {p[k]:.2f}{mark}")

### scikit-learn: те же слова и символьные n-граммы

`LogisticRegression` на тех же бинарных признаках — это та же модель, только с другим оптимизатором и другой силой регуляризации, поэтому на спорных репликах ответы могут отличаться. Вторая модель использует символьные n-граммы (2–4 символа внутри слов): «отмените», «отменить», «отмена» делят n-граммы «отм», «мен», поэтому незнакомая форма слова всё равно узнаётся. Для языков с богатой морфологией это особенно важно.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

X_text_tr = [s for s, _ in train]
y_text_tr = [c for _, c in train]
X_text_te = [s for s, _ in test]
y_text_te = [c for _, c in test]

models = {
    "слова (LogisticRegression)": make_pipeline(
        CountVectorizer(tokenizer=bag, token_pattern=None, lowercase=False, binary=True),
        LogisticRegression(C=10, max_iter=2000)),
    "символьные n-граммы (LogisticRegression)": make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4), sublinear_tf=True),
        LogisticRegression(C=10, max_iter=2000)),
}
pred = {}
for name, m in models.items():
    m.fit(X_text_tr, y_text_tr)
    pred[name] = m.predict(X_text_te)
    print(f"{name:<34} точность = {accuracy_score(y_text_te, pred[name]):.2f}")

print()
for s, gold, a, c in zip(X_text_te, y_text_te, pred["слова (LogisticRegression)"], pred["символьные n-граммы (LogisticRegression)"]):
    print(f"{s:<52} {gold:<11} {a:<11} {c:<11}")

intent_clf = models["символьные n-граммы (LogisticRegression)"]
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(y_text_te, pred["символьные n-граммы (LogisticRegression)"], labels=INTENTS,
                                        ax=ax, colorbar=False, xticks_rotation=45)
ax.set_title("Интенты: символьные n-граммы")
ax.set_xlabel("предсказано")
ax.set_ylabel("верно")
plt.tight_layout()
plt.show()

## 3. Заполнение слотов

### Правила

Самый простой слот-филлер — словари и шаблоны: список городов (газеттир), слова для дат и времени, числительные. **Роль** города (откуда или куда) в русском определяет предлог: «**из** Ташкента» → `from`, «**в** Самарканд», «**до** Бухары» → `to`. Обратите внимание, что газеттир хранит основы («бухар»), чтобы узнавать все падежные формы.


Оценка по **спанам**: слот засчитывается, только если совпали и тип, и границы (как в NER). Ниже — оценка правил на тестовых репликах и их ошибки: например, правила не знают города «Андижан», которого нет в газеттире.

In [ ]:
# газеттир городов: основа -> название в именительном падеже
CITIES = {"ташкент": "Ташкент", "самарканд": "Самарканд", "бухар": "Бухара", "хив": "Хива",
          "нукус": "Нукус", "карши": "Карши", "термез": "Термез"}
FROM_PREP = {"из", "от"}
TO_PREP = {"в", "во", "до"}
DATE_WORDS = {"сегодня", "завтра", "послезавтра", "понедельник", "вторник", "среду", "четверг",
              "пятницу", "субботу", "воскресенье"}
MONTHS = {"января", "февраля", "марта", "апреля", "мая", "июня", "июля", "августа",
          "сентября", "октября", "ноября", "декабря"}
TIME_WORDS = {"утром", "днём", "днем", "вечером", "ночью"}
COUNT_WORDS = {"один": 1, "одного": 1, "одна": 1, "два": 2, "две": 2, "двух": 2, "двоих": 2,
               "три": 3, "трёх": 3, "троих": 3, "четыре": 4, "четырёх": 4, "четверых": 4,
               "пять": 5, "пятерых": 5, "шесть": 6, "шестерых": 6}

def city_of(tok):
    low = tok.lower()
    for stem_, name in CITIES.items():
        if low.startswith(stem_):
            return name
    return None

def rule_tags(toks):
    tags = ["O"] * len(toks)
    low = [t.lower() for t in toks]
    unassigned = []
    for i, w in enumerate(low):
        if re.fullmatch(r"\d{1,2}:\d{2}", w) or w in TIME_WORDS:
            tags[i] = "B-time"
        elif w.isdigit() and i + 1 < len(toks) and low[i + 1] in MONTHS:
            tags[i], tags[i + 1] = "B-date", "I-date"           # «10 мая»
        elif w in DATE_WORDS:
            tags[i] = "B-date"
        elif w in COUNT_WORDS:
            tags[i] = "B-count"
        elif city_of(w):
            prev = low[i - 1] if i > 0 else ""
            if prev in FROM_PREP:
                tags[i] = "B-from"
            elif prev in TO_PREP:
                tags[i] = "B-to"
            else:
                unassigned.append(i)
    # города без предлога («Ташкент – Нукус»): первый — откуда, второй — куда
    for k, i in enumerate(unassigned):
        tags[i] = "B-from" if k == 0 and len(unassigned) > 1 else "B-to"
    return tags

def normalize(slot, value):
    # приводим значение слота к каноническому виду
    if slot in ("from", "to"):
        return city_of(value) or value
    if slot == "count":
        return COUNT_WORDS.get(value.lower(), int(value) if value.isdigit() else value)
    return value.lower()

for s in ["Хочу уехать из Самарканда в Бухару 10 мая вечером, два билета", "Ташкент – Нукус на завтра в 8:00"]:
    toks = tokenize(s)
    print(" ".join(f"{t}/{x}" if x != "O" else t for t, x in zip(toks, rule_tags(toks))))


def spans(tags):
    # BIO-теги -> множество спанов (слот, начало, конец)
    out, cur = [], None
    for i, t in enumerate(tags + ["O"]):
        if cur and not (t.startswith("I-") and t[2:] == cur[0]):
            out.append((cur[0], cur[1], i)); cur = None
        if t.startswith("B-") or (t.startswith("I-") and cur is None):
            cur = (t[2:], i)
    return set(out)

def span_prf(gold_tags, pred_tags):
    tp = fp = fn = 0
    for g, p in zip(gold_tags, pred_tags):
        G, P = spans(list(g)), spans(list(p))
        tp += len(G & P); fp += len(P - G); fn += len(G - P)
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return prec, rec, f1

rule_pred = [rule_tags(toks) for toks, _ in test_bio]
gold = [tags for _, tags in test_bio]
print("правила: P = %.2f  R = %.2f  F1 = %.2f" % span_prf(gold, rule_pred))
for (toks, g), p in zip(test_bio, rule_pred):
    if spans(g) != spans(p):
        print("  ", " ".join(f"{t}/{x}" if x != "O" else t for t, x in zip(toks, p)))
        print("   верно:", " ".join(f"{t}/{x}" if x != "O" else t for t, x in zip(toks, g)))

### CRF: слот-филлер, который учится по разметке

Линейно-цепочечный CRF (тема 9) оценивает всю последовательность тегов сразу: признаки токена (само слово, окончание, заглавная буква, соседние слова) плюс веса переходов между тегами: например, переход `O → I-date` (продолжение даты без её начала) получает штраф. Обучение — на тех же 60 размеченных репликах. Ниже — качество на тесте, а затем самые сильные признаки для каждого тега и самые «запрещённые» и «разрешённые» переходы.

In [ ]:
import sklearn_crfsuite

def word_features(toks, i):
    w = toks[i]
    f = {
        "bias": 1.0,
        "w": w.lower(),
        "suf2": w.lower()[-2:],
        "suf3": w.lower()[-3:],
        "pre4": w.lower()[:4],
        "title": w[:1].isupper(),
        "first": i == 0,
        "digit": w[:1].isdigit(),
        "has_colon": ":" in w,
    }
    f["prev"] = toks[i - 1].lower() if i > 0 else "<s>"
    f["next"] = toks[i + 1].lower() if i + 1 < len(toks) else "</s>"
    return f

def sent_features(toks):
    return [word_features(toks, i) for i in range(len(toks))]

crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.05, c2=0.05, max_iterations=200,
                           all_possible_transitions=True)
crf.fit([sent_features(t) for t, _ in train_bio], [tags for _, tags in train_bio])

crf_pred = [list(p) for p in crf.predict([sent_features(t) for t, _ in test_bio])]
print("CRF:     P = %.2f  R = %.2f  F1 = %.2f" % span_prf(gold, crf_pred))
print("правила: P = %.2f  R = %.2f  F1 = %.2f\n" % span_prf(gold, rule_pred))
for (toks, g), p, r in zip(test_bio, crf_pred, rule_pred):
    print(" ".join(f"{t}/{x}" if x != "O" else t for t, x in zip(toks, p)),
          "" if spans(g) == spans(p) else "   <-- CRF ошибся")


from collections import Counter
for label in ["B-from", "B-to", "B-date", "B-count"]:
    feats = [(f, w) for (f, l), w in crf.state_features_.items() if l == label]
    top = sorted(feats, key=lambda t: -t[1])[:6]
    print(f"{label:<8}", ", ".join(f"{f} ({w:.1f})" for f, w in top))

print("\nПереходы между тегами (самые отрицательные и самые положительные веса):")
trans = sorted(crf.transition_features_.items(), key=lambda t: t[1])
for (a, b_), w in trans[:3] + trans[-3:]:
    print(f"   {a:>7} -> {b_:<7} {w:6.2f}")

Чему научился CRF? Для `B-from` и `B-to` модель опирается на предыдущее слово-предлог (`prev:из`, `prev:в`, `prev:до`): именно его мы вручную вписали в правила.

Сравните ошибки двух подходов. CRF нашёл незнакомый город «Андижан» по контексту (предлог «в» + заглавная буква), а правила — нет, его нет в газеттире. Зато правила знают из словаря «один» и «понедельник», которых не было в 60 обучающих репликах, и CRF их пропустил. Обычно выигрывает комбинация: признак «слово есть в газеттире» добавляют в CRF.


### Модуль понимания (NLU) целиком

Соединим всё: интент — символьная модель scikit-learn, слоты — правила (или CRF), значения слотов нормализуются: «Бухары» → «Бухара», «двоих» → 2.

In [ ]:
def extract_slots(toks, tags):
    # BIO -> [(слот, нормализованное значение)]
    out = []
    for name, i, j in sorted(spans(tags), key=lambda s: s[1]):
        out.append((name, normalize(name, " ".join(toks[i:j]))))
    return out

def nlu(text, tagger="rules"):
    toks = tokenize(text)
    proba = intent_clf.predict_proba([text])[0]
    k = int(proba.argmax())
    tags = rule_tags(toks) if tagger == "rules" else list(crf.predict([sent_features(toks)])[0])
    return str(intent_clf.classes_[k]), round(float(proba[k]), 2), extract_slots(toks, tags)

for s in ["Хочу два билета из Бухары в Ташкент на пятницу", "Столик на двоих сегодня в 19:30", "Отмените, пожалуйста"]:
    print(s)
    print("   ", nlu(s), "\n")

## 4. Фреймовый менеджер диалога

**Фрейм** — набор слотов, которые нужно заполнить для задачи: для билета `from, to, date, count`, для столика `date, time, count`. Менеджер — конечный автомат: состояния `START → ASK_слот → … → CONFIRM → DONE`. На каждом шаге он обновляет фрейм слотами из реплики и спрашивает первый незаполненный слот. В состоянии `CONFIRM` «да» завершает заказ, а «нет» с новым значением исправляет слот. Интент `cancel` сбрасывает фрейм в любом состоянии.

**Шаблонная генерация (NLG)** превращает действие менеджера в текст. Для русского шаблона нужна морфология: «из Ташкент**а** в Бухар**у**» (падежи), «2 билет**а**», «5 билет**ов**» (согласование с числительным).

In [ ]:
# Шаблонная генерация ответа (template NLG). Русскому шаблону нужны падежи и согласование с числом!
CITY_FORMS = {  # название -> (родительный «из …», винительный «в …»)
    "Ташкент": ("Ташкента", "Ташкент"), "Самарканд": ("Самарканда", "Самарканд"),
    "Бухара": ("Бухары", "Бухару"), "Хива": ("Хивы", "Хиву"), "Нукус": ("Нукуса", "Нукус"),
    "Карши": ("Карши", "Карши"), "Термез": ("Термеза", "Термез"),
}
YES = {"да", "верно", "подтверждаю", "конечно"}
NO = {"нет", "неверно", "не"}

def plural(n, forms):
    # 1 билет, 2 билета, 5 билетов; 11–14 -> «билетов»
    if isinstance(n, int):
        if n % 10 == 1 and n % 100 != 11:
            return f"{n} {forms[0]}"
        if 2 <= n % 10 <= 4 and not 12 <= n % 100 <= 14:
            return f"{n} {forms[1]}"
    return f"{n} {forms[2]}"

def at_time(t):
    return "в " + t if re.fullmatch(r"\d{1,2}:\d{2}", t) else t     # «в 19:00», но просто «вечером»

ASK = {
    ("buy_ticket", "from"): "Откуда вы поедете?",
    ("buy_ticket", "to"): "Куда вы поедете?",
    ("buy_ticket", "date"): "На какой день нужен билет?",
    ("buy_ticket", "count"): "Сколько нужно билетов?",
    ("book_table", "date"): "На какой день забронировать столик?",
    ("book_table", "time"): "На какое время?",
    ("book_table", "count"): "На сколько человек?",
}

def describe(task, f):
    if task == "buy_ticket":
        return (f"{plural(f['count'], ('билет', 'билета', 'билетов'))} из {CITY_FORMS[f['from']][0]} "
                f"в {CITY_FORMS[f['to']][1]} на {f['date']}")
    return f"столик на {f['date']} {at_time(f['time'])}, {plural(f['count'], ('человек', 'человека', 'человек'))}"

def nlg(action, task=None, arg=None):
    if action == "ask":
        return ASK[(task, arg)]
    if action == "confirm":
        return f"Итак: {describe(task, arg)}. Всё верно?"
    if action == "done":
        return f"Готово! Оформлено: {describe(task, arg)}."
    return {
        "greet": "Здравствуйте! Я могу продать билет на поезд или забронировать столик.",
        "what_change": "Что нужно изменить?",
        "cancelled": "Хорошо, заказ отменён.",
        "bye": "Всего доброго!",
        "fallback": "Извините, я умею только продавать билеты и бронировать столики.",
    }[action]

print(nlg("confirm", "buy_ticket", {"from": "Бухара", "to": "Хива", "date": "пятницу", "count": 1}))
print(nlg("confirm", "buy_ticket", {"from": "Ташкент", "to": "Бухара", "date": "завтра", "count": 3}))
print(nlg("confirm", "book_table", {"date": "субботу", "time": "вечером", "count": 5}))

print()

dialogue_1 = [
    "Здравствуйте",
    "Хочу купить билет в Самарканд на завтра",
    "Из Ташкента",
    "Два билета",
    "Нет, лучше в Бухару",
    "Да",
    "Спасибо, до свидания",
]
dialogue_2 = [
    "Нужен столик на троих",
    "На субботу",
    "В 19:00",
    "Отмените, планы изменились",
]


REQUIRED = {"buy_ticket": ["from", "to", "date", "count"],
            "book_table": ["date", "time", "count"]}

class FrameDM:
    # фреймовый менеджер: состояние, активная задача, заполненные слоты
    def __init__(self, nlu):
        self.nlu = nlu
        self.reset()
        self.path = ["START"]

    def reset(self):
        self.task, self.frame, self.state = None, {}, "START"

    def go(self, state):
        self.state = state
        self.path.append(state)

    def fill(self, slots):
        for name, value in slots:
            if name in REQUIRED[self.task]:
                self.frame[name] = value

    def next_question(self):
        missing = [s for s in REQUIRED[self.task] if s not in self.frame]
        if missing:
            self.go("ASK_" + missing[0])
            return nlg("ask", self.task, missing[0])
        self.go("CONFIRM")
        return nlg("confirm", self.task, self.frame)

    def respond(self, text):
        intent, conf, slots = self.nlu(text)
        words = {w.lower() for w in tokenize(text)}
        if self.state == "CONFIRM":                       # ждём подтверждения: «да» -> DONE, «нет» + новый слот -> исправить
            if words & YES:
                self.go("DONE")
                return nlg("done", self.task, self.frame)
            if words & NO or slots:
                self.fill(slots)
                return nlg("what_change") if not slots else self.next_question()
        if intent == "cancel" and conf > 0.4:
            self.reset(); self.go("START")
            return nlg("cancelled")
        if intent == "thanks_bye" and not slots:
            self.go("END")
            return nlg("bye")
        if self.task is None or self.state == "DONE":
            if intent in REQUIRED:
                self.reset(); self.task = intent
            elif intent == "greeting":
                return nlg("greet")
            else:
                return nlg("fallback")
        self.fill(slots)
        return self.next_question()

def run_dialogue(user_turns, nlu_fn=nlu):
    dm = FrameDM(nlu_fn)
    for u in user_turns:
        reply = dm.respond(u)
        print(f"Клиент:   {u}")
        print(f"Бот:    {reply}")
        print(f"          [{dm.state}] {dm.task} {dm.frame}\n")
    return dm

dm1 = run_dialogue(dialogue_1)

Второй диалог: бронирование столика, которое пользователь отменяет. Затем нарисуем пройденные состояния автомата.

In [ ]:
dm2 = run_dialogue(dialogue_2)


import networkx as nx

pos = {"START": (0, 0), "ASK_from": (1, 1.5), "ASK_to": (2, 1.5), "ASK_date": (3, 1.5),
       "ASK_time": (3, -1.5), "ASK_count": (4, 0), "CONFIRM": (5, 0), "DONE": (6, 0.8), "END": (7, 0)}
G = nx.DiGraph()
G.add_nodes_from(pos)
edges1 = list(zip(dm1.path, dm1.path[1:]))
edges2 = list(zip(dm2.path, dm2.path[1:]))
G.add_edges_from(edges1 + edges2)
fig, ax = plt.subplots(figsize=(11, 4))
visited = set(dm1.path) | set(dm2.path)
nx.draw_networkx_nodes(G, pos, node_size=1900, ax=ax,
                       node_color=["#F2C14E" if n in visited else "#DDDDDD" for n in G])
nx.draw_networkx_labels(G, pos, font_size=8, ax=ax)
for edges, color, rad in [(edges1, "#2B55C9", 0.15), (edges2, "#D9822B", -0.15)]:
    e = [x for x in dict.fromkeys(edges) if x[0] != x[1]]
    nx.draw_networkx_edges(G, pos, edgelist=e, edge_color=color, width=2, ax=ax, node_size=1900,
                           arrowsize=16, connectionstyle=f"arc3,rad={rad}")
ax.plot([], [], color="#2B55C9", label="диалог 1 (билет)")
ax.plot([], [], color="#D9822B", label="диалог 2 (столик, отмена)")
ax.legend(loc="lower left")
ax.set_title("Состояния, пройденные в двух диалогах")
ax.axis("off")
plt.tight_layout()
plt.show()

## 5. Сценарий против генерации

Сценарная система полностью управляема: она говорит только то, что есть во фрейме, но звучит однообразно. Генеративная модель пишет ответ сама. Покажем проблему на игрушечной биграммной модели, обученной на 15 ответах системы с конкретными городами и датами (в трёх ответах дата не упомянута) (температура $T$ меняет распределение, как в теме 16).

1. **Обычная генерация**: модель не видит фрейма и подставляет города, которые часто встречались в обучении, — «галлюцинирует» значения слотов.
2. **Делексикализация**: в обучающих ответах значения заменены метками `FROM`, `TO`, `DATE`. Модель генерирует шаблон, а значения подставляются из фрейма в нужной форме. Ошибочных значений не бывает, но модель может пропустить слот.

Это игрушка, не настоящая модель: она показывает только компромисс «гладкость против контроля».

In [ ]:
# ответы системы с конкретными значениями слотов — «обучающий корпус» игрушечной генеративной модели
responses = [
    "Билет из Ташкента в Самарканд на завтра оформлен.",
    "Ваш билет из Бухары в Ташкент на пятницу готов.",
    "Готово, билет из Самарканда в Бухару на субботу оформлен.",
    "Хорошо, оформляю билет из Ташкента в Хиву на завтра.",
    "Билет из Нукуса в Ташкент на понедельник оформлен, хорошей поездки!",
    "Ваш билет из Ташкента в Бухару на сегодня готов, хорошей поездки!",
    "Отлично, билет из Хивы в Самарканд на завтра оформлен.",
    "Хорошо, билет из Самарканда в Ташкент на пятницу готов.",
    "Готово, ваш билет из Бухары в Нукус на субботу оформлен.",
    "Билет из Ташкента в Нукус на сегодня готов.",
    "Оформляю ваш билет из Хивы в Ташкент на понедельник.",
    "Ваш билет из Самарканда в Хиву на завтра готов, хорошей поездки!",
    "Ваш билет из Бухары в Ташкент готов.",
    "Билет из Самарканда в Хиву оформлен, хорошей поездки!",
    "Готово, билет из Ташкента в Нукус оформлен.",
]
# формы слов -> (слот, нормальная форма); «из» требует родительного, «в» — винительного падежа
SLOT_FORMS = {
    "from": {"Ташкента": "Ташкент", "Самарканда": "Самарканд", "Бухары": "Бухара", "Хивы": "Хива", "Нукуса": "Нукус"},
    "to": {"Ташкент": "Ташкент", "Самарканд": "Самарканд", "Бухару": "Бухара", "Хиву": "Хива", "Нукус": "Нукус"},
    "date": {w: w for w in ["завтра", "сегодня", "пятницу", "субботу", "понедельник"]},
}

def delexicalize(s):
    # заменяем значения слотов на метки: «из Ташкента» -> «из FROM»
    out = []
    for t in tokenize(s):
        for name in ["from", "to", "date"]:
            if t in SLOT_FORMS[name]:
                t = name.upper(); break
        out.append(t)
    return " ".join(out)

def lexicalize(s, frame):
    # обратная операция: метки -> нужные формы из фрейма
    return (s.replace("FROM", CITY_FORMS[frame["from"]][0]).replace("TO", CITY_FORMS[frame["to"]][1])
             .replace("DATE", frame["date"]))

frame_demo = {"from": "Ташкент", "to": "Бухара", "date": "завтра", "count": 2}
print(len(responses), "ответов")


from collections import defaultdict

def train_bigram(sentences):
    counts = defaultdict(lambda: defaultdict(int))
    for s in sentences:
        toks = ["<s>"] + tokenize(s) + ["</s>"]
        for a, b_ in zip(toks, toks[1:]):
            counts[a][b_] += 1
    return counts

def sample(counts, T=1.0, rng=None, max_len=25):
    out, cur = [], "<s>"
    while len(out) < max_len:
        words = list(counts[cur])
        logits = np.log(np.array([counts[cur][w] for w in words], dtype=float))
        p = np.exp((logits - logits.max()) / T)
        p /= p.sum()
        cur = words[rng.choice(len(words), p=p)]
        if cur == "</s>":
            break
        out.append(cur)
    return " ".join(out)

def mentioned(text):
    # какие значения слотов упомянуты в ответе
    found = {}
    for t in tokenize(text):
        for name in ["from", "to", "date"]:
            if t in SLOT_FORMS[name]:
                found.setdefault(name, []).append(SLOT_FORMS[name][t])
    return found

def check(text, frame):
    m = mentioned(text)
    wrong = any(v != frame[k] for k, vs in m.items() for v in vs)
    missing = any(k not in m for k in ["from", "to", "date"])
    return wrong, missing

raw_lm = train_bigram(responses)
delex_lm = train_bigram([delexicalize(r) for r in responses])
print("делексикализованный ответ:", delexicalize(responses[0]))
print()

rng = np.random.default_rng(1)
print("фрейм:", frame_demo)
for T in [0.5, 1.0, 2.0]:
    print(f"\nT = {T}")
    for _ in range(3):
        print("   обычная:  ", sample(raw_lm, T, rng))
    for _ in range(2):
        print("   делекс.:", lexicalize(sample(delex_lm, T, rng), frame_demo))

Посчитаем по 300 ответов на каждую температуру: доля ответов с **неверным** значением слота и доля ответов, где какой-то слот **пропущен**.

In [ ]:
rows = []
for T in [0.5, 1.0, 1.5, 2.0]:
    r = np.array([check(sample(raw_lm, T, rng), frame_demo) for _ in range(300)])
    d = np.array([check(lexicalize(sample(delex_lm, T, rng), frame_demo), frame_demo) for _ in range(300)])
    rows.append((T, r[:, 0].mean(), r[:, 1].mean(), d[:, 0].mean(), d[:, 1].mean()))
    print(f"T={T}:  обычная неверное значение {r[:, 0].mean():.0%}, пропущен слот {r[:, 1].mean():.0%}   |   "
          f"делекс. неверное значение {d[:, 0].mean():.0%}, пропущен слот {d[:, 1].mean():.0%}")
tpl = nlg("done", "buy_ticket", frame_demo)
print(f"\nшаблон: {tpl}  ->  неверное значение/пропущен слот: {check(tpl, frame_demo)}")

rows = np.array(rows)
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(rows[:, 0], rows[:, 1], "o-", label="обычная: неверное значение")
ax.plot(rows[:, 0], rows[:, 4], "s--", label="делекс.: пропущен слот")
ax.plot(rows[:, 0], rows[:, 3], "^-", label="делекс.: неверное значение")
ax.set_xlabel("температура T")
ax.set_ylim(-0.03, 1.03)
ax.set_title("Генеративные ответы: ошибки по температуре")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Как это выглядит на настоящей модели

Современные системы часто генерируют ответ большой языковой моделью, передавая ей фрейм в подсказке. Ниже — `Qwen/Qwen2.5-0.5B-Instruct` (0,5 млрд параметров, ≈1 ГБ) получает фрейм в JSON и пишет ответ клиенту. Посмотрите, все ли значения из фрейма попали в ответ и не добавила ли модель лишнего (цену, время, номер поезда). В настоящих системах ответ LLM проверяют по фрейму или оставляют модели только «перефразирование» шаблона. Ячейка работает только в Colab с интернетом; на CPU генерация занимает около минуты.

In [ ]:
# в Colab transformers и torch уже установлены
try:
    import json, torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    name = "Qwen/Qwen2.5-0.5B-Instruct"                # ≈1 ГБ, скачивается один раз
    tok = AutoTokenizer.from_pretrained(name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = "auto" if device == "cuda" else torch.float32     # на CPU float32 быстрее, чем bfloat16
    llm = AutoModelForCausalLM.from_pretrained(name, torch_dtype=dtype).to(device)
    messages = [
        {"role": "system", "content": "Ты вежливый ассистент железнодорожной кассы. Ответь клиенту одним предложением по-русски, что билет оформлен. Используй только данные из фрейма."},
        {"role": "user", "content": "Фрейм заказа: " + json.dumps(frame_demo, ensure_ascii=False)},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok([text], return_tensors="pt").to(device)
    torch.manual_seed(0)
    for k in range(4):
        out = llm.generate(**inputs, max_new_tokens=60, do_sample=True, temperature=0.9, top_p=0.95)
        reply = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        print(f"{k + 1}. {reply}")
        print("   упомянутые значения слотов:", mentioned(reply), "\n")
except Exception as e:
    print("Не удалось запустить модель (нужен интернет в Colab):", e)

## 6. Задания

1. **Новый интент.** Добавьте интент `price` («Сколько стоит билет до Хивы?») с 8–10 репликами и разметкой слотов. Переобучите обе модели scikit-learn. Какие реплики из других интентов теперь путаются с `price`?
2. **Новый слот.** Добавьте в разметку слот `class` (эконом, бизнес) в 5–6 реплик `buy_ticket`, переобучите CRF и проверьте его на «Билет бизнес-класса до Самарканда на завтра».
3. **Абляция признаков CRF.** Уберите из `word_features` признаки `prev` и `next`, затем `suf2`/`suf3`. Как меняется F1 на тесте? Какие признаки важнее для русского и почему (подсказка: предлоги)?
4. **Менеджер.** Сейчас ответ «Самарканд» без предлога на вопрос «Куда вы поедете?» правила помечают как `to` случайно. Сделайте так, чтобы менеджер передавал в NLU ожидаемый слот (`self.state`) и город без предлога записывался в спрошенный слот. Проверьте на диалоге, где пользователь отвечает «Ташкент» на вопрос «Откуда?».
5. **Генерация с проверкой.** Напишите функцию, которая генерирует делексикализованный ответ, проверяет, что в нём есть все три метки `FROM`, `TO`, `DATE`, и при неудаче генерирует заново (не больше 20 попыток). Сколько попыток в среднем нужно при $T = 1$ и при $T = 2$? Почему такой фильтр невозможен для обычной (не делексикализованной) модели без знания фрейма?

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь